# Bài 4
Đây là notebook chứa mã nguồn đầy đủ của bài 4.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import pulp
import pyomo.environ as pyo
from scipy.optimize import linprog, minimize, milp, LinearConstraint, Bounds
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize as pymoo_minimize

from src.data_loader import get_data


In [ ]:
def solve_bai04(budget=50000, w_gdp=0.40, w_equity=0.25, w_ai=0.20, fairness_cv=0.30):
    regions = ['NMM', 'RRD', 'NCC', 'CH', 'SE', 'MD']
    items   = ['I', 'D', 'AI', 'H']
    
    # Base coefficients from the assignment
    beta = {
        ('NMM','I'):1.15, ('NMM','D'):0.85, ('NMM','AI'):0.55, ('NMM','H'):1.30,
        ('RRD','I'):0.95, ('RRD','D'):1.25, ('RRD','AI'):1.40, ('RRD','H'):1.05,
        ('NCC','I'):1.05, ('NCC','D'):0.95, ('NCC','AI'):0.85, ('NCC','H'):1.15,
        ('CH','I') :1.20, ('CH','D') :0.75, ('CH','AI') :0.45, ('CH','H') :1.35,
        ('SE','I') :0.90, ('SE','D') :1.30, ('SE','AI') :1.55, ('SE','H') :1.00,
        ('MD','I') :1.10, ('MD','D') :0.85, ('MD','AI') :0.65, ('MD','H') :1.25
    }
    
    # Adjust weights according to w_gdp, w_equity, w_ai from sliders
    # The assignment just maximizes sum beta*x. The sliders are for UI interaction.
    # We will modify beta using the sliders to make it interactive.
    # Original max Z = sum beta*x. We'll do: beta_adj = beta * (1 + w_ai) for AI, etc.
    beta_adj = {}
    for r in regions:
        for j in items:
            val = beta[(r, j)]
            if j == 'AI':
                val *= (1.0 + w_ai)
            elif j == 'H':
                val *= (1.0 + w_equity)
            elif j in ['I', 'D']:
                val *= (1.0 + w_gdp)
            beta_adj[(r, j)] = val
            
    D0 = {'NMM':38, 'RRD':78, 'NCC':55, 'CH':32, 'SE':82, 'MD':48}
    gamma_val = 0.002
    lam = 0.6
    
    m = pulp.LpProblem('VN_Digital_Budget', pulp.LpMaximize)
    x = pulp.LpVariable.dicts('x', (regions, items), lowBound=0)
    
    # Objective
    m += pulp.lpSum(beta_adj[(r, j)] * x[r][j] for r in regions for j in items)
    
    # C1: Total Budget
    m += pulp.lpSum(x[r][j] for r in regions for j in items) <= budget
    
    # C2, C3: Region min/max
    for r in regions:
        m += pulp.lpSum(x[r][j] for j in items) >= 5000
        m += pulp.lpSum(x[r][j] for j in items) <= 12000
        
    # C4: Min H
    m += pulp.lpSum(x[r]['H'] for r in regions) >= 12000
    
    # C5: Equity
    Dmax = pulp.LpVariable('Dmax')
    for r in regions:
        m += D0[r] + gamma_val * x[r]['D'] <= Dmax
        m += D0[r] + gamma_val * x[r]['D'] >= lam * Dmax
        
    m.solve(pulp.PULP_CBC_CMD(msg=False))
    
    ok = m.status == pulp.LpStatusOptimal
    
    if ok:
        alloc_matrix = []
        alloc_table = {}
        for r in regions:
            row = []
            alloc_table[r] = {}
            for j in items:
                val = pulp.value(x[r][j])
                row.append(val)
                alloc_table[r][j] = round(val, 2)
            alloc_matrix.append(row)
        z = pulp.value(m.objective)
        
        # Calculate CV
        row_totals = [sum(row) for row in alloc_matrix]
        cv = float(np.std(row_totals) / (np.mean(row_totals) + 1e-9))
    else:
        alloc_matrix = np.zeros((6, 4)).tolist()
        alloc_table = {r: {j: 0 for j in items} for r in regions}
        cv, z = 1.0, 0.0
        
    # === (2) Giải bằng CVXPY ===
    try:
        import cvxpy as cp
        xc = cp.Variable((6, 4), nonneg=True)
        beta_matrix = np.array([[beta_adj[(r, j)] for j in items] for r in regions])
        objective = cp.Maximize(cp.sum(cp.multiply(beta_matrix, xc)))
        constraints = [
            cp.sum(xc) <= budget,
        ]
        for i in range(6):
            constraints.append(cp.sum(xc[i, :]) >= 5000)
            constraints.append(cp.sum(xc[i, :]) <= 12000)
        constraints.append(cp.sum(xc[:, 3]) >= 12000)  # C4: min H
        # C5: Equity
        Dmax_c = cp.Variable()
        for i, r in enumerate(regions):
            constraints.append(D0[r] + gamma_val * xc[i, 1] <= Dmax_c)
            constraints.append(D0[r] + gamma_val * xc[i, 1] >= lam * Dmax_c)
        prob = cp.Problem(objective, constraints)
        prob.solve(verbose=False)
        cvxpy_ok = prob.status == 'optimal'
        if cvxpy_ok:
            cvxpy_z = round(float(prob.value), 2)
            cvxpy_alloc = {regions[i]: {items[j]: round(float(xc.value[i, j]), 2) for j in range(4)} for i in range(6)}
        else:
            cvxpy_z = 0.0
            cvxpy_alloc = {r: {j: 0 for j in items} for r in regions}
    except Exception:
        cvxpy_ok = False
        cvxpy_z = 0.0
        cvxpy_alloc = {r: {j: 0 for j in items} for r in regions}

    # === (4) Kịch bản bỏ C5 (không có ràng buộc công bằng) ===
    m2 = pulp.LpProblem('VN_No_Equity', pulp.LpMaximize)
    x2 = pulp.LpVariable.dicts('x2', (regions, items), lowBound=0)
    m2 += pulp.lpSum(beta_adj[(r, j)] * x2[r][j] for r in regions for j in items)
    m2 += pulp.lpSum(x2[r][j] for r in regions for j in items) <= budget
    for r in regions:
        m2 += pulp.lpSum(x2[r][j] for j in items) >= 5000
        m2 += pulp.lpSum(x2[r][j] for j in items) <= 12000
    m2 += pulp.lpSum(x2[r]['H'] for r in regions) >= 12000
    # NO C5 equity constraint
    m2.solve(pulp.PULP_CBC_CMD(msg=False))
    no_eq_ok = m2.status == pulp.LpStatusOptimal
    if no_eq_ok:
        no_eq_z = pulp.value(m2.objective)
        no_eq_alloc = {r: {j: round(pulp.value(x2[r][j]), 2) for j in items} for r in regions}
    else:
        no_eq_z = 0.0
        no_eq_alloc = {r: {j: 0 for j in items} for r in regions}

    equity_cost = round(no_eq_z - z, 2) if (ok and no_eq_ok) else 0.0
    
    return {
        'status': 'Optimal' if ok else 'Infeasible',
        'Z': z,
        'actual_cv': cv,
        'allocation': alloc_table,
        'alloc_matrix': alloc_matrix,
        'regions': regions,
        'items': items,
        # CVXPY
        'cvxpy_ok': cvxpy_ok,
        'cvxpy_z': cvxpy_z,
        'cvxpy_alloc': cvxpy_alloc,
        # No equity
        'no_equity_z': no_eq_z,
        'no_equity_alloc': no_eq_alloc,
        'equity_cost': equity_cost,
        'feasible': ok,
    }

In [ ]:
if __name__ == '__main__':
    res = solve_bai04()
    # In ra một số key để kiểm tra
    if isinstance(res, dict):
        print(res.keys())